In [ ]:
import csv
import requests
from bs4 import BeautifulSoup
import re

def find_names():

    BASE_URL = "https://arxiv.org/"

    headers = {
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                            "(KHTML, like Gecko) Chrome/120.0 Safari/537.36",
                "X-Requested-With": "XMLHttpRequest",
                "Referer": BASE_URL,
                "Accept": "*/*",
            }

    TIMEOUT = 15

    response = requests.get(
        BASE_URL,
        headers=headers,
        timeout=TIMEOUT,
    )

    soup = BeautifulSoup(response.text, 'html.parser')
    physics_block = soup.select('h2 + ul > li')

    link_list = []
    link_list_act = []

    for block in physics_block:
        link_block = block.find('a')
        link = link_block.get('href')
        if link == 'https://info.arxiv.org/help/cs/index.html':
            break
        link_list.append(link)

    for i in link_list:
        s = re.sub(r'/.+/', '', i)
        link_list_act.append(s)
    
    return link_list_act

find_names()


In [ ]:
art_list = {}

def extract_htmls(l):
    
    BASE_URL = f"https://arxiv.org/list/{l}/2026-11?skip=0&show=1000"

    headers = {
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                            "(KHTML, like Gecko) Chrome/120.0 Safari/537.36",
                "X-Requested-With": "XMLHttpRequest",
                "Referer": BASE_URL,
                "Accept": "*/*",
            }

    TIMEOUT = 15

    response = requests.get(
        BASE_URL,
        headers=headers,
        timeout=TIMEOUT,
    )

    links = []

    soup = BeautifulSoup(response.text, 'html.parser')
    art_blocks = soup.find_all('dt')
    for block in art_blocks:
        link = block.find('a', title='View HTML')
        if link:
            links.append(link.get('href'))
        else:
            continue
    art_list[l] = links
    return art_list, len(art_list)

for each in find_names():
    extract_htmls(each)

#extract_texts('nucl-ex')

In [ ]:
art_list_act = {}

for key, value in art_list.items():
    if key == 'math':
        break
    art_list_act[key] = value[0:11]


In [ ]:
def extract_texts(arx):

    all_article = []

    BASE_URL = arx

    headers = {
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                            "(KHTML, like Gecko) Chrome/120.0 Safari/537.36",
                "X-Requested-With": "XMLHttpRequest",
                "Referer": BASE_URL,
                "Accept": "*/*",
            }

    TIMEOUT = 15

    response = requests.get(
        BASE_URL,
        headers=headers,
        timeout=TIMEOUT,
    )

    soup = BeautifulSoup(response.text, 'html.parser')
    head = soup.find('h1')
    if head:
        for formula in head.find_all('math'):
            formula.replace_with(' [FORMULA] ')
        all_article.append(' '.join(head.get_text().split()))
    abstract = soup.find('div', id='abstract1')
    if abstract:
        p = abstract.find('p')
        if p:
            for formula in p.find_all('math'):
                formula.replace_with(' [FORMULA] ')
            abstract_text = ' '.join(p.get_text().split())
            all_article.append(abstract_text)
    #bib = soup.find('section', class_='ltx_bibliography')
    text_blocks = soup.find_all('div', id=re.compile(r'^p\d+$'))
    if text_blocks:
        for block in text_blocks:
            if abstract and abstract in block.parents:
                continue
            p1 = block.find('p')
            if p1:
                for formula in p1.find_all('math'):
                    formula.replace_with(' [FORMULA] ')
                article_text = ' '.join(p1.get_text().split())
                all_article.append(article_text)
    if not text_blocks:
        text_blocks = soup.find_all('section', class_='ltx_section')
        if text_blocks:
            for block in text_blocks:
                head2 = block.find('h2')
                if head2:
                    for formula in head2.find_all('math'):
                        formula.replace_with(' [FORMULA] ')
                    all_article.append(' '.join(head2.get_text().split()))
                for p2 in block.find_all('p'):
                    for formula in p2.find_all('math'):
                        formula.replace_with(' [FORMULA] ')
                    article_text = ' '.join(p2.get_text().split())
                    all_article.append(article_text)



    return all_article


#extract_texts('https://arxiv.org/html/2611.12086v1')

In [ ]:
from tqdm.auto import tqdm
import time

texts = {}

for key, urls in art_list_act.items():
    texts[key] = []
    for url in tqdm(urls, desc=key, leave=False):
        try:
            s = extract_texts(url)
            if len(s) < 9:
                continue
            texts[key].append(s)
        except requests.RequestException as e:
            print(f'Ошибка {url}: {e}')
        time.sleep(1)


In [ ]:
counter = 0

for key, value in texts.items():
    for v in value:
        counter += 1

counter

In [ ]:
add = {'astro-ph': ['https://arxiv.org/html/2611.00121v1',
  'https://arxiv.org/html/2611.00132v1',
  'https://arxiv.org/html/2611.00150v1',
  'https://arxiv.org/html/2611.00160v1',
  'https://arxiv.org/html/2611.00188v1',
  'https://arxiv.org/html/2611.00190v2',
  'https://arxiv.org/html/2611.00195v1',
  'https://arxiv.org/html/2611.00198v1',
  'https://arxiv.org/html/2611.00199v1',
  'https://arxiv.org/html/2611.00202v1',
  'https://arxiv.org/html/2611.00205v1',
  'https://arxiv.org/html/2611.00207v1',
  'https://arxiv.org/html/2611.00208v1',
  'https://arxiv.org/html/2611.00212v1',
  'https://arxiv.org/html/2611.00215v2']}

for key, urls in add.items():
    texts[key] = []
    for url in tqdm(urls, desc=key, leave=False):
        try:
            s = extract_texts(url)
            if len(s) < 9:
                continue
            texts[key].append(s)
        except requests.RequestException as e:
            print(f'Ошибка {url}: {e}')
        time.sleep(1)

In [ ]:
counter = 0

for key, value in texts.items():
    for v in value:
        counter += 1

counter

In [ ]:
texts_act = {}

for key, value in texts.items():
    texts_act[key] = []
    for v in value:
        al = ' '.join(v)
        if 'In the age of big data and AI: do we still need chemical bonds?' in al:
            continue
        texts_act[key].append(al)


In [ ]:
# adding maths 

maths = {'maths':['https://arxiv.org/html/2611.03682v1',
  'https://arxiv.org/html/2611.03688v1',
  'https://arxiv.org/html/2611.03693v1',
  'https://arxiv.org/html/2611.03695v1',
  'https://arxiv.org/html/2611.03701v1',
  'https://arxiv.org/html/2611.03711v2',
  'https://arxiv.org/html/2611.03714v1',
  'https://arxiv.org/html/2611.03715v1',
  'https://arxiv.org/html/2611.03716v1',
  'https://arxiv.org/html/2611.03732v1',
  'https://arxiv.org/html/2611.03733v1',
  'https://arxiv.org/html/2611.03736v1',
  'https://arxiv.org/html/2611.03745v1',
  'https://arxiv.org/html/2611.03750v1',
  'https://arxiv.org/html/2611.03758v1',
  'https://arxiv.org/html/2611.03763v2',
  'https://arxiv.org/html/2611.03773v1',
  'https://arxiv.org/html/2611.03778v1',
  'https://arxiv.org/html/2611.03797v1',
  'https://arxiv.org/html/2611.03823v1',
  'https://arxiv.org/html/2611.03830v1',
  'https://arxiv.org/html/2611.03835v1',
  'https://arxiv.org/html/2611.03840v1',
  'https://arxiv.org/html/2611.03843v1',
  'https://arxiv.org/html/2611.03847v1',
  'https://arxiv.org/html/2611.03855v1',
  'https://arxiv.org/html/2611.03856v1',
  'https://arxiv.org/html/2611.03859v1',
  'https://arxiv.org/html/2611.03860v2',
  'https://arxiv.org/html/2611.03863v1',
  'https://arxiv.org/html/2611.03866v1',
  'https://arxiv.org/html/2611.03872v1',
  'https://arxiv.org/html/2611.03878v1',
  'https://arxiv.org/html/2611.03879v1',
  'https://arxiv.org/html/2611.03883v1',
  'https://arxiv.org/html/2611.03884v1',
  'https://arxiv.org/html/2611.03886v1',
  'https://arxiv.org/html/2611.03889v1',
  'https://arxiv.org/html/2611.03890v1',
  'https://arxiv.org/html/2611.03895v1',
  'https://arxiv.org/html/2611.03896v1',
  'https://arxiv.org/html/2611.03901v1',
  'https://arxiv.org/html/2611.03905v1',
  'https://arxiv.org/html/2611.03907v1',
  'https://arxiv.org/html/2611.03915v1',
  'https://arxiv.org/html/2611.03920v1']}

for key, urls in maths.items():
    texts[key] = []
    for url in tqdm(urls, desc=key, leave=False):
        try:
            s = extract_texts(url)
            if len(s) < 9:
                continue
            texts[key].append(s)
        except requests.RequestException as e:
            print(f'Ошибка {url}: {e}')
        time.sleep(1)


In [ ]:
c = 0

for key, value in texts.items():
    if key == 'maths':
        for v in value:
            if c == 10:
                break
            al = ' '.join(v)
            texts_act[key].append(al)
            c += 1



In [ ]:
def extract_htmls_cs():
    
    BASE_URL = f"https://arxiv.org/list/cs/new"

    headers = {
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                            "(KHTML, like Gecko) Chrome/120.0 Safari/537.36",
                "X-Requested-With": "XMLHttpRequest",
                "Referer": BASE_URL,
                "Accept": "*/*",
            }

    TIMEOUT = 15

    response = requests.get(
        BASE_URL,
        headers=headers,
        timeout=TIMEOUT,
    )

    cs_list = {}

    soup = BeautifulSoup(response.text, 'html.parser')
    art_blocks = soup.find_all('dt')
    for block in art_blocks:
        link = block.find('a', title='View HTML')
        if not link:
            continue
        dd = block.find_next_sibling('dd')                       
        subject = dd.find('span', class_='primary-subject') if dd else None
        if not subject:
            continue
        key = subject.get_text(strip=True)                       
        cs_list.setdefault(key, []).append(link.get('href'))

    return cs_list


extract_htmls_cs()
#extract_texts('nucl-ex')

In [ ]:
check = {}

cc = 0
LIMIT = 200
CL_KEY = 'Computation and Language (cs.CL)'
CL_LIMIT = 10
CL_KEY2 = 'Software Engineering (cs.SE)'
CL_LIMIT2 = 20
CL_KEY3 = 'Computer Vision and Pattern Recognition (cs.CV)'
CL_LIMIT3 = 10
CL_KEY4 = 'Machine Learning (cs.LG)'
CL_LIMIT4 = 20
CL_KEY5 = 'Artificial Intelligence (cs.AI)'
CL_LIMIT5 = 20
CL_KEY6 = 'Robotics (cs.RO)'
CL_LIMIT6 = 20


for key, urls in extract_htmls_cs().items():
    if cc >= LIMIT:
        break
    if key == CL_KEY:
        urls = urls[:CL_LIMIT]
    if key == CL_KEY2:
        urls = urls[:CL_LIMIT2]
    if key == CL_KEY3:
        urls = urls[:CL_LIMIT3]
    if key == CL_KEY4:
        urls = urls[:CL_LIMIT4]
    if key == CL_KEY5:
        urls = urls[:CL_LIMIT5]
    if key == CL_KEY6:
        urls = urls[:CL_LIMIT6]
    check[key] = []
    for url in tqdm(urls, desc=key, leave=False):
        if cc >= LIMIT:
            break
        cc += 1
        try:
            s = extract_texts(url)
            if len(s) < 9:
                continue
            check[key].append(s)
        except requests.RequestException as e:
            print(f'Ошибка {url}: {e}')
        time.sleep(1)

In [ ]:
count = 0
LIMIT = 100

for key, value in check.items():
    if count >= LIMIT:
        break
    texts_act[key] = []
    for v in value:
        if count >= LIMIT:
            break
        texts_act[key].append(' '.join(v))
        count += 1

In [ ]:
rename = {
    'astro-ph': 'Astrophysics',
    'cond-mat': 'Condensed Matter',
    'gr-qc': 'General Relativity and Quantum Cosmology',
    'hep-ex': 'High Energy Physics - Experiment',
    'hep-lat': 'High Energy Physics - Lattice',
    'hep-ph': 'High Energy Physics - Phenomenology',
    'hep-th': 'High Energy Physics - Theory',
    'math-ph': 'Mathematical Physics',
    'nlin': 'Nonlinear Sciences',
    'nucl-ex': 'Nuclear Experiment',
    'nucl-th': 'Nuclear Theory',
    'physics': 'Physics',
    'quant-ph': 'Quantum Physics',
    'maths': 'Mathematics'
        
}

texts_act = {rename.get(k, k): v for k, v in texts_act.items()}

In [ ]:
import json

with open('ph_maths_it1.json', 'w', encoding='utf-8') as f:
    json.dump(texts_act, f, ensure_ascii=False, indent=2)